# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [2]:
%help

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 



# Available Magic Commands

## Sessions Magic

----
    %help                             Return a list of descriptions and input types for all magic commands. 
    %profile            String        Specify a profile in your aws configuration to use as the credentials provider.
    %region             String        Specify the AWS region in which to initialize a session. 
                                      Default from ~/.aws/config on Linux or macOS, 
                                      or C:\Users\ USERNAME \.aws\config" on Windows.
    %idle_timeout       Int           The number of minutes of inactivity after which a session will timeout. 
                                      Default: 2880 minutes (48 hours).
    %timeout            Int           The number of minutes after which a session will timeout. 
                                      Default: 2880 minutes (48 hours).
    %session_id_prefix  String        Define a String that will precede all session IDs in the format 
                                      [session_id_prefix]-[session_id]. If a session ID is not provided,
                                      a random UUID will be generated.
    %status                           Returns the status of the current Glue session including its duration, 
                                      configuration and executing user / role.
    %session_id                       Returns the session ID for the running session.
    %list_sessions                    Lists all currently running sessions by ID.
    %stop_session                     Stops the current session.
    %glue_version       String        The version of Glue to be used by this session. 
                                      Currently, the only valid options are 2.0, 3.0 and 4.0. 
                                      Default: 2.0.
    %reconnect          String        Specify a live session ID to switch/reconnect to the sessions.
----

## Selecting Session Types

----
    %streaming          String        Sets the session type to Glue Streaming.
    %etl                String        Sets the session type to Glue ETL.
    %session_type       String        Specify a session_type to be used. Supported values: streaming and etl.
----

## Glue Config Magic 
*(common across all session types)*

----

    %%configure         Dictionary    A json-formatted dictionary consisting of all configuration parameters for 
                                      a session. Each parameter can be specified here or through individual magics.
    %iam_role           String        Specify an IAM role ARN to execute your session with.
                                      Default from ~/.aws/config on Linux or macOS, 
                                      or C:\Users\%USERNAME%\.aws\config` on Windows.
    %number_of_workers  int           The number of workers of a defined worker_type that are allocated 
                                      when a session runs.
                                      Default: 5.
    %additional_python_modules  List  Comma separated list of additional Python modules to include in your cluster 
                                      (can be from Pypi or S3).
    %%tags        Dictionary          Specify a json-formatted dictionary consisting of tags to use in the session.
    
    %%assume_role Dictionary, String  Specify a json-formatted dictionary or an IAM role ARN string to create a session 
                                      for cross account access.
                                      E.g. {valid arn}
                                      %%assume_role 
                                      'arn:aws:iam::XXXXXXXXXXXX:role/AWSGlueServiceRole' 
                                      E.g. {credentials}
                                      %%assume_role
                                      {
                                            "aws_access_key_id" : "XXXXXXXXXXXX",
                                            "aws_secret_access_key" : "XXXXXXXXXXXX",
                                            "aws_session_token" : "XXXXXXXXXXXX"
                                       }
----

                                      
## Magic for Spark Sessions (ETL & Streaming)

----
    %worker_type        String        Set the type of instances the session will use as workers. 
    %connections        List          Specify a comma separated list of connections to use in the session.
    %extra_py_files     List          Comma separated list of additional Python files From S3.
    %extra_jars         List          Comma separated list of additional Jars to include in the cluster.
    %spark_conf         String        Specify custom spark configurations for your session. 
                                      E.g. %spark_conf spark.serializer=org.apache.spark.serializer.KryoSerializer
----

## Action Magic

----

    %%sql               String        Run SQL code. All lines after the initial %%sql magic will be passed
                                      as part of the SQL code.  
    %matplot      Matplotlib figure   Visualize your data using the matplotlib library.
                                      E.g. 
                                      import matplotlib.pyplot as plt
                                      # Set X-axis and Y-axis values
                                      x = [5, 2, 8, 4, 9]
                                      y = [10, 4, 8, 5, 2]
                                      # Create a bar chart 
                                      plt.bar(x, y) 
                                      # Show the plot
                                      %matplot plt    
    %plotly            Plotly figure  Visualize your data using the plotly library.
                                      E.g.
                                      import plotly.express as px
                                      #Create a graphical figure
                                      fig = px.line(x=["a","b","c"], y=[1,3,2], title="sample figure")
                                      #Show the figure
                                      %plotly fig

  
                
----



## Cell 1: Setup

In [9]:
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5
%idle_timeout 30
%iam_role arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel
%additional_python_modules pyarrow

Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Current iam_role is arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel
iam_role has been set to arn:aws:iam::311414083183:role/AWSGlueServiceRole-CyberThreatIntel.
Additional python modules to be included:
pyarrow


## Cell 2: Imports and Paths

In [1]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

bucket = "cyber-threat-intel-data-lake"
silver_base = f"s3://{bucket}/silver"
gold_base = f"s3://{bucket}/gold"

run_date = (datetime.now(ZoneInfo("Africa/Cairo")) - timedelta(days=1)).strftime("%Y-%m-%d")

paths = {
    # Silver sources
    "assets_silver": f"{silver_base}/asset_inventory_clean/",
    "cisa_silver": f"{silver_base}/cisa_kev_clean/",
    "epss_silver": f"{silver_base}/epss_scores/",
    "cves_silver": f"{silver_base}/cves_clean/",

    # Gold Parquet targets
    "dim_date": f"{gold_base}/dim_date/",
    "dim_asset": f"{gold_base}/dim_asset/",
    "dim_cve": f"{gold_base}/dim_cve/",
    "fact_asset_vulnerability": f"{gold_base}/fact_asset_vulnerability/",
    "fact_epss_trend": f"{gold_base}/fact_epss_trend/",

    # Gold CSV targets
    "dim_date_csv": f"{gold_base}/dim_date_csv/",
    "dim_asset_csv": f"{gold_base}/dim_asset_csv/",
    "dim_cve_csv": f"{gold_base}/dim_cve_csv/",
    "fact_asset_vulnerability_csv": f"{gold_base}/fact_asset_vulnerability_csv/",
    "fact_epss_trend_csv": f"{gold_base}/fact_epss_trend_csv/",
}

print("Gold layer paths configured.")
print("Run date:", run_date)

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 30
Session ID: eb618c49-e170-4011-9df3-2c52834ad88c
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--additional-python-modules pyarrow
Waiting for session eb618c49-e170-4011-9df3-2c52834ad88c to get into ready status...
Session eb618c49-e170-4011-9df3-2c52834ad88c has been created.
Gold layer paths configured.
Run date: 2026-06-09


## Cell 3: Helper Functions

In [2]:
def read_silver(path):
    """Read silver Parquet, return None if not exists."""
    try:
        df = spark.read.parquet(path)
        print(f"Read {df.count()} rows from {path}")
        return df
    except Exception as e:
        print(f"No silver data at {path}: {e}")
        return None


def write_gold(df, parquet_path, csv_path):
    """Write gold as single Parquet file and single CSV file (coalesce(1))."""
    
    count = df.count()
    
    # Single Parquet file
    df.coalesce(1).write.mode("overwrite").parquet(parquet_path)
    print(f"[PARQUET] Wrote {count} rows to {parquet_path}")
    
    # Single CSV file
    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("nullValue", "") \
        .option("dateFormat", "yyyy-MM-dd") \
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
        .csv(csv_path)
    
    print(f"[CSV] Wrote {count} rows to {csv_path}")


def generate_date_dimension(start_year=1999, end_year=2030):
    """Generate standard date dimension as single DataFrame."""
    
    date_df = spark.range(0, (end_year - start_year + 1) * 366).select(
        F.expr(f"date_add(make_date({start_year}, 1, 1), cast(id as int))").alias("full_date")
    ).filter(
        F.col("full_date") <= F.make_date(F.lit(end_year), F.lit(12), F.lit(31))
    )

    dim_date = date_df.select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("full_date").alias("full_date"),
        F.year("full_date").alias("year"),
        F.month("full_date").alias("month"),
        F.dayofmonth("full_date").alias("day"),
        F.quarter("full_date").alias("quarter"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        (F.dayofweek("full_date").isin([1, 7])).alias("is_weekend"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.date_format("full_date", "yyyy-MM").alias("year_month"),
    )

    return dim_date

### Cell 4: Build DimDate

In [3]:
dim_date = generate_date_dimension(start_year=1999, end_year=2030)
write_gold(dim_date, paths["dim_date"], paths["dim_date_csv"])
dim_date.show(5)

[PARQUET] Wrote 11688 rows to s3://cyber-threat-intel-data-lake/gold/dim_date/
[CSV] Wrote 11688 rows to s3://cyber-threat-intel-data-lake/gold/dim_date_csv/
+--------+----------+----+-----+---+-------+-----------+--------+----------+----------+------------+----------+
|date_key| full_date|year|month|day|quarter|day_of_week|day_name|month_name|is_weekend|week_of_year|year_month|
+--------+----------+----+-----+---+-------+-----------+--------+----------+----------+------------+----------+
|19990101|1999-01-01|1999|    1|  1|      1|          6|  Friday|   January|     false|          53|   1999-01|
|19990102|1999-01-02|1999|    1|  2|      1|          7|Saturday|   January|      true|          53|   1999-01|
|19990103|1999-01-03|1999|    1|  3|      1|          1|  Sunday|   January|      true|          53|   1999-01|
|19990104|1999-01-04|1999|    1|  4|      1|          2|  Monday|   January|     false|           1|   1999-01|
|19990105|1999-01-05|1999|    1|  5|      1|          3| T

## Cell 5: Build DimAsset (SCD Type 2)

In [4]:
assets_df = read_silver(paths["assets_silver"])

if assets_df is None:
    raise ValueError("Asset inventory silver data is required.")

assets_df = assets_df.select([
    F.col(c).alias(c.strip().lower().replace(" ", "_"))
    for c in assets_df.columns
])

today = datetime.now().strftime("%Y-%m-%d")

dim_asset = assets_df.select(
    F.monotonically_increasing_id().cast("long").alias("asset_key"),
    F.col("asset_id"),
    F.col("hostname"),
    F.col("vendor"),
    F.col("product"),
    F.col("version"),
    F.col("criticality"),
    F.col("internet_facing"),
    F.lit(today).cast("date").alias("effective_start_date"),
    F.lit(None).cast("date").alias("effective_end_date"),
    F.lit(True).alias("is_current"),
)

write_gold(dim_asset, paths["dim_asset"], paths["dim_asset_csv"])
dim_asset.show(5, truncate=False)

Read 5000 rows from s3://cyber-threat-intel-data-lake/silver/asset_inventory_clean/
[PARQUET] Wrote 5000 rows to s3://cyber-threat-intel-data-lake/gold/dim_asset/
[CSV] Wrote 5000 rows to s3://cyber-threat-intel-data-lake/gold/dim_asset_csv/
+---------+--------+---------------------------+---------+---------------+-------+-----------+---------------+--------------------+------------------+----------+
|asset_key|asset_id|hostname                   |vendor   |product        |version|criticality|internet_facing|effective_start_date|effective_end_date|is_current|
+---------+--------+---------------------------+---------+---------------+-------+-----------+---------------+--------------------+------------------+----------+
|0        |A00001  |vmware-esxi-1              |vmware   |esxi           |9.2.1  |Low        |No             |2026-06-10          |null              |true      |
|1        |A00002  |microsoft-exchange-server-2|microsoft|exchange server|1.3.29 |High       |No             |

## Cell 6: Build DimCVE (SCD Type 0)

In [5]:
cves_df = read_silver(paths["cves_silver"])

if cves_df is None:
    raise ValueError("CVEs silver data is required.")

cves_df = cves_df.select([
    F.col(c).alias(c.strip().lower().replace(" ", "_"))
    for c in cves_df.columns
])

dim_cve = cves_df.select(
    F.monotonically_increasing_id().cast("long").alias("cve_key"),
    F.col("cve_id"),
    F.col("description"),
    F.to_date(F.col("published_date")).alias("published_date"),
    F.col("cvss_version"),
    F.col("vector_string"),
    F.col("basescore").cast("double").alias("base_score"),
    F.col("baseseverity").alias("base_severity"),
    F.col("exploitabilityscore").cast("double").alias("exploitability_score"),
    F.col("impactscore").cast("double").alias("impact_score"),
    F.col("criteria"),
    F.col("matchcriteriaid").alias("match_criteria_id"),
    F.col("vendor").alias("cpe_vendor"),
    F.col("product").alias("cpe_product"),
    F.col("version").alias("cpe_version"),
)

dim_cve = dim_cve.dropDuplicates(["cve_id"])

write_gold(dim_cve, paths["dim_cve"], paths["dim_cve_csv"])
print(f"DimCVE count: {dim_cve.count()}")
dim_cve.show(3, truncate=False)

Read 355726 rows from s3://cyber-threat-intel-data-lake/silver/cves_clean/
[PARQUET] Wrote 355726 rows to s3://cyber-threat-intel-data-lake/gold/dim_cve/
[CSV] Wrote 355726 rows to s3://cyber-threat-intel-data-lake/gold/dim_cve_csv/
DimCVE count: 355726
+-----------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+------------+--------------------------+----------+-------------+--------------------+------------+--------------------------------------+------------------------------------+----------+-----------+-----------+
|cve_key    |cve_id       |description                                                                                                                                                                   |published_date|cvss_version|vector_string             |base_score|base_severity|exploitability_score|impact_score|cr

## Cell 7: Build FactEPPSTrend


In [6]:
epss_df = read_silver(paths["epss_silver"])

if epss_df is None:
    raise ValueError("EPSS silver data is required.")

if "acve_id" in epss_df.columns and "cve_id" not in epss_df.columns:
    epss_df = epss_df.withColumnRenamed("acve_id", "cve_id")

dim_cve_for_join = spark.read.parquet(paths["dim_cve"]).select("cve_key", "cve_id")

fact_epss_trend = epss_df.join(
    dim_cve_for_join,
    on="cve_id",
    how="inner"
).select(
    F.col("cve_key"),
    F.date_format(F.col("score_date"), "yyyyMMdd").cast("int").alias("date_key"),
    F.col("epss_score"),
    F.col("percentile").alias("epss_percentile"),
    F.col("score_date"),
)

fact_epss_trend = fact_epss_trend.dropDuplicates(["cve_key", "date_key"])

write_gold(fact_epss_trend, paths["fact_epss_trend"], paths["fact_epss_trend_csv"])
print(f"FactEPPSTrend count: {fact_epss_trend.count()}")
fact_epss_trend.show(5)

Read 1354201 rows from s3://cyber-threat-intel-data-lake/silver/epss_scores/
[PARQUET] Wrote 1352728 rows to s3://cyber-threat-intel-data-lake/gold/fact_epss_trend/
[CSV] Wrote 1352728 rows to s3://cyber-threat-intel-data-lake/gold/fact_epss_trend_csv/
FactEPPSTrend count: 1352728
+------------+--------+----------+---------------+----------+
|     cve_key|date_key|epss_score|epss_percentile|score_date|
+------------+--------+----------+---------------+----------+
|120259106601|20260605|    7.4E-4|        0.22579|2026-06-05|
| 85899367944|20260605|    3.6E-4|         0.1116|2026-06-05|
|120259106210|20260605|    3.9E-4|        0.12091|2026-06-05|
|111669171016|20260605|    3.5E-4|        0.10774|2026-06-05|
| 17179889611|20260605|    6.5E-4|        0.20361|2026-06-05|
+------------+--------+----------+---------------+----------+
only showing top 5 rows


## Cell 8: Build FactAssetVulnerability

In [7]:
dim_asset = spark.read.parquet(paths["dim_asset"]).filter(F.col("is_current") == True)
dim_cve = spark.read.parquet(paths["dim_cve"])
cisa_df = read_silver(paths["cisa_silver"])

if cisa_df is not None:
    cisa_df = cisa_df.select([
        F.col(c).alias(c.strip().lower().replace(" ", "_"))
        for c in cisa_df.columns
    ])
    cisa_flag = cisa_df.select(
        F.col("cve_id"),
        F.lit(1).alias("is_cisa_exploited"),
        F.col("date_added").alias("cisa_date_added"),
        F.col("due_date").alias("cisa_due_date"),
        F.col("known_ransomware_campaign_use").alias("cisa_ransomware_flag"),
    )
else:
    cisa_flag = spark.createDataFrame(
        [],
        "cve_id string, is_cisa_exploited int, cisa_date_added date, cisa_due_date date, cisa_ransomware_flag string"
    )

latest_epss = spark.read.parquet(paths["fact_epss_trend"]) \
    .withColumn("score_date", F.to_date(F.col("score_date").cast("string"), "yyyy-MM-dd")) \
    .groupBy("cve_key") \
    .agg(
        F.max("score_date").alias("latest_score_date"),
        F.first("epss_score").alias("latest_epss_score"),
        F.first("epss_percentile").alias("latest_epss_percentile")
    )

asset_normalized = dim_asset.select(
    "asset_key", "asset_id", "hostname", "vendor", "product", "version",
    "criticality", "internet_facing"
).withColumn("match_vendor", F.lower(F.trim(F.col("vendor")))) \
 .withColumn("match_product", F.lower(F.trim(F.col("product"))))

cve_normalized = dim_cve.select(
    "cve_key", "cve_id", "cpe_vendor", "cpe_product", "cpe_version",
    "base_score", "base_severity", "exploitability_score", "impact_score"
).withColumn("match_cpe_vendor", F.lower(F.trim(F.col("cpe_vendor")))) \
 .withColumn("match_cpe_product", F.lower(F.trim(F.col("cpe_product")))) \
 .withColumn("cpe_version_clean", F.lower(F.trim(F.col("cpe_version"))))

matched = asset_normalized.join(
    cve_normalized,
    on=(
        (F.col("match_vendor") == F.col("match_cpe_vendor")) &
        (F.col("match_product") == F.col("match_cpe_product")) &
        (
            (F.col("cpe_version_clean") == "*") |
            (F.col("cpe_version_clean") == F.col("version")) |
            (F.col("cpe_version_clean").isNull())
        )
    ),
    how="inner"
)

matched = matched.withColumn(
    "match_confidence",
    F.when(F.col("cpe_version_clean") == F.col("version"), F.lit("Exact"))
     .when(F.col("cpe_version_clean") == "*", F.lit("Wildcard"))
     .when(F.col("cpe_version_clean").isNull(), F.lit("VendorOnly"))
     .otherwise(F.lit("Unknown"))
)

matched = matched.join(cisa_flag, on="cve_id", how="left")
matched = matched.withColumn("is_cisa_exploited", F.coalesce(F.col("is_cisa_exploited"), F.lit(0)))

matched = matched.join(
    latest_epss.select("cve_key", "latest_epss_score", "latest_epss_percentile"),
    on="cve_key",
    how="left"
)

today_key = int(datetime.now().strftime("%Y%m%d"))

fact_asset_vulnerability = matched.select(
    F.col("asset_key"),
    F.col("cve_key"),
    F.lit(today_key).alias("date_key"),
    F.col("base_score"),
    F.col("exploitability_score"),
    F.col("impact_score"),
    F.coalesce(F.col("latest_epss_score"), F.lit(0.0)).alias("epss_score"),
    F.coalesce(F.col("latest_epss_percentile"), F.lit(0.0)).alias("epss_percentile"),
    F.col("is_cisa_exploited"),
    F.when(F.col("internet_facing") == "Yes", F.lit(1)).otherwise(F.lit(0)).alias("is_internet_facing"),
    F.col("match_confidence"),
    F.when(F.col("criticality") == "Critical", F.lit(4))
     .when(F.col("criticality") == "High", F.lit(3))
     .when(F.col("criticality") == "Medium", F.lit(2))
     .otherwise(F.lit(1)).alias("criticality_weight"),
)

dim_cve_dates = dim_cve.select("cve_key", "published_date")
fact_asset_vulnerability = fact_asset_vulnerability.join(dim_cve_dates, on="cve_key", how="left")

fact_asset_vulnerability = fact_asset_vulnerability.withColumn(
    "days_since_published",
    F.datediff(F.current_date(), F.col("published_date"))
).drop("published_date")

fact_asset_vulnerability = fact_asset_vulnerability.withColumn(
    "risk_score",
    (F.col("base_score") / 10.0 * 0.30) +
    (F.col("epss_score") * 100.0 * 0.30) +
    (F.col("is_cisa_exploited") * 10.0 * 0.20) +
    (F.col("criticality_weight") / 4.0 * 10.0 * 0.15) +
    (F.col("is_internet_facing") * 10.0 * 0.05)
).withColumn("risk_score", F.round(F.col("risk_score"), 2))

fact_asset_vulnerability = fact_asset_vulnerability.select(
    "asset_key", "cve_key", "date_key",
    "base_score", "exploitability_score", "impact_score",
    "epss_score", "epss_percentile",
    "is_cisa_exploited", "is_internet_facing",
    "match_confidence", "criticality_weight",
    "risk_score", "days_since_published"
)

write_gold(fact_asset_vulnerability, paths["fact_asset_vulnerability"], paths["fact_asset_vulnerability_csv"])
print(f"FactAssetVulnerability count: {fact_asset_vulnerability.count()}")
fact_asset_vulnerability.show(5)

Read 1614 rows from s3://cyber-threat-intel-data-lake/silver/cisa_kev_clean/
[PARQUET] Wrote 38887 rows to s3://cyber-threat-intel-data-lake/gold/fact_asset_vulnerability/
[CSV] Wrote 38887 rows to s3://cyber-threat-intel-data-lake/gold/fact_asset_vulnerability_csv/
FactAssetVulnerability count: 38887
+---------+-----------+--------+----------+--------------------+------------+----------+---------------+-----------------+------------------+----------------+------------------+----------+--------------------+
|asset_key|    cve_key|date_key|base_score|exploitability_score|impact_score|epss_score|epss_percentile|is_cisa_exploited|is_internet_facing|match_confidence|criticality_weight|risk_score|days_since_published|
+---------+-----------+--------+----------+--------------------+------------+----------+---------------+-----------------+------------------+----------------+------------------+----------+--------------------+
|     4999|25769809030|20260610|       5.9|                 2.2|   

## Cell 9: Validation and Summary

In [8]:
gold_tables = {
    "dim_date": (paths["dim_date"], paths["dim_date_csv"]),
    "dim_asset": (paths["dim_asset"], paths["dim_asset_csv"]),
    "dim_cve": (paths["dim_cve"], paths["dim_cve_csv"]),
    "fact_epss_trend": (paths["fact_epss_trend"], paths["fact_epss_trend_csv"]),
    "fact_asset_vulnerability": (paths["fact_asset_vulnerability"], paths["fact_asset_vulnerability_csv"]),
}

print("=" * 70)
print("GOLD LAYER VALIDATION — SINGLE FILE PARQUET + CSV")
print("=" * 70)

for name, (parquet_path, csv_path) in gold_tables.items():
    print(f"\n--- {name} ---")
    
    try:
        df_pq = spark.read.parquet(parquet_path)
        print(f"[PARQUET] {df_pq.count()} rows, {len(df_pq.columns)} columns")
    except Exception as e:
        print(f"[PARQUET] FAILED: {e}")
    
    try:
        df_csv = spark.read.option("header", "true").csv(csv_path)
        print(f"[CSV] {df_csv.count()} rows, {len(df_csv.columns)} columns")
    except Exception as e:
        print(f"[CSV] FAILED: {e}")

print("\n" + "=" * 70)
print("SAMPLE QUERIES")
print("=" * 70)

print("\nTop 10 Risk Scores:")
spark.read.parquet(paths["fact_asset_vulnerability"]) \
    .orderBy(F.desc("risk_score")) \
    .limit(10) \
    .show(truncate=False)

print("\nEPSS Trend Sample:")
spark.read.parquet(paths["fact_epss_trend"]) \
    .filter(F.col("cve_key").isNotNull()) \
    .orderBy(F.desc("score_date")) \
    .limit(10) \
    .show(truncate=False)

GOLD LAYER VALIDATION — SINGLE FILE PARQUET + CSV

--- dim_date ---
[PARQUET] 11688 rows, 12 columns
[CSV] 11688 rows, 12 columns

--- dim_asset ---
[PARQUET] 5000 rows, 11 columns
[CSV] 5000 rows, 11 columns

--- dim_cve ---
[PARQUET] 355726 rows, 15 columns
[CSV] 386555 rows, 15 columns

--- fact_epss_trend ---
[PARQUET] 1352728 rows, 5 columns
[CSV] 1352728 rows, 5 columns

--- fact_asset_vulnerability ---
[PARQUET] 38887 rows, 14 columns
[CSV] 38887 rows, 14 columns

SAMPLE QUERIES

Top 10 Risk Scores:
+---------+-----------+--------+----------+--------------------+------------+----------+---------------+-----------------+------------------+----------------+------------------+----------+--------------------+
|asset_key|cve_key    |date_key|base_score|exploitability_score|impact_score|epss_score|epss_percentile|is_cisa_exploited|is_internet_facing|match_confidence|criticality_weight|risk_score|days_since_published|
+---------+-----------+--------+----------+--------------------+----